# Hestia P4 — larger-model evidence and review

This notebook reads **32 actual Colab GPU generations**, without rerunning inference.
Use `person4_large_models_colab.ipynb` to reproduce the model runs. The evidence
includes fixed shared prompts, model revisions/configuration, raw outputs, timing,
schema validation, and assistant-assisted content reviews. Earlier small-model
experiments remain separate. P1 owns the final 15-model notebook and category plan.


In [1]:
from pathlib import Path
import importlib.util, json
import pandas as pd
from IPython.display import display, Markdown
ROOT=Path.cwd()
if not (ROOT/'manage.py').exists():
    ROOT=ROOT.parent
spec=importlib.util.spec_from_file_location('verify_p4_large',ROOT/'docs/a6-person-4/verify_large_results.py')
checker=importlib.util.module_from_spec(spec)
spec.loader.exec_module(checker)
rows, metadata=checker.verify(ROOT/'a6_results/person4_large')
results=pd.DataFrame(rows)
results['generation_seconds']=pd.to_numeric(results['generation_seconds'])
results['quality_score_0_to_10']=pd.to_numeric(results['quality_score_0_to_10'])
assert len(results)==32 and results['quality_score_0_to_10'].notna().all()
print(json.dumps(metadata,indent=2))


E:\Hestia\a6-brain-dump\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tiiuae/Falcon3-7B-Instruct runs= 8 schema_valid= 7 mean_generation_s= 17.453 reviewed= 8
Qwen/Qwen2.5-14B-Instruct runs= 8 schema_valid= 8 mean_generation_s= 20.778 reviewed= 8
microsoft/Phi-3-medium-4k-instruct runs= 8 schema_valid= 3 mean_generation_s= 33.919 reviewed= 8
Qwen/Qwen3-14B runs= 8 schema_valid= 8 mean_generation_s= 21.801 reviewed= 8
Verified 32 unique actual generations; 32/32 have recorded content reviews.
{
  "models": {
    "tiiuae/Falcon3-7B-Instruct": {
      "revision": "1e57a0ecd176c7c139f289c60a74e57f887c3dfb",
      "status": "complete",
      "qwen3_thinking": null,
      "load_seconds": 298.308,
      "footprint_gib": 6.097,
      "last_completed": "BD2/many",
      "peak_allocated_gib": 6.726,
      "completed_generations": 8
    },
    "Qwen/Qwen2.5-14B-Instruct": {
      "revision": "cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8",
      "status": "complete",
      "qwen3_thinking": null,
      "load_seconds": 557.104,
      "footprint_gib": 11.956,
      "last_

In [2]:
display(Markdown((ROOT/'docs/a6-person-4/large-model-findings.md').read_text(encoding='utf-8')))
display(results[['model_id','input_id','shot_mode','generation_seconds','schema_valid','quality_score_0_to_10']])


# P4 larger-model findings

Actual CUDA inference on Tesla T4. NF4 weights, FP32 computation/non-quantized modules, eager attention, 220 new tokens maximum, greedy generation; Qwen3 thinking disabled.

| Model | Runs | Schema valid | Mean generation | Mean reviewed quality |
| --- | ---: | ---: | ---: | ---: |
| tiiuae/Falcon3-7B-Instruct | 8 | 7/8 | 17.453 s | 8.500/10 |
| Qwen/Qwen2.5-14B-Instruct | 8 | 8/8 | 20.778 s | 9.500/10 |
| microsoft/Phi-3-medium-4k-instruct | 8 | 3/8 | 33.919 s | 6.875/10 |
| Qwen/Qwen3-14B | 8 | 8/8 | 21.801 s | 9.250/10 |

| Model | Zero-shot quality | One-shot | Few-shot | Many-shot |
| --- | ---: | ---: | ---: | ---: |
| tiiuae/Falcon3-7B-Instruct | 7.50 | 8.50 | 9.00 | 9.00 |
| Qwen/Qwen2.5-14B-Instruct | 9.50 | 9.50 | 9.50 | 9.50 |
| microsoft/Phi-3-medium-4k-instruct | 4.50 | 7.50 | 7.50 | 8.00 |
| Qwen/Qwen3-14B | 9.50 | 9.50 | 9.00 | 9.00 |

Observed content limitations:

- Falcon extracted the concerns consistently but often overstated tap/Friday urgency; its BD2 zero-shot JSON was truncated at 220 tokens.
- Qwen2.5 produced concise, schema-valid outputs in every case. Guest preparation was often only MEDIUM; valid JSON still needs priority review.
- Phi changed the required schema in both zero-shot cases, misclassified library returns, invented overdue status, and truncated all three example-based BD2 outputs. Its 3/8 schema rate is conditional on the same 220-token cap used for every model.
- Qwen3 was schema-valid in every case but overstated leak urgency in early modes and under-ranked detector batteries in few/many-shot. More examples did not monotonically improve quality.

Scores are assistant-assisted content judgments using extraction/types/priority/clarity/format (3/2/2/2/1). Inspect each raw response and its explicit criterion notes in the notebook. Independent human approval is not asserted.

Load timing includes uncached downloads. Generation timing is CUDA-synchronized. GPU/quantized timing cannot be treated as a controlled speed comparison against the earlier CPU/native measurements or a remote paid API.

These candidates cover one large and three above-10B positions under the original ranges. Main PR #31 now uses different balanced bands and retains the earlier small-model selection. P1/the team owns whether to retain that plan or consolidate this larger-model evidence; do not blindly concatenate both P4 sets. See large-models.md for the exact handoff.


,model_id,input_id,shot_mode,generation_seconds,schema_valid,quality_score_0_to_10
0,tiiuae/Falcon3-7B-Instruct,BD1,zero,17.783,True,8
1,tiiuae/Falcon3-7B-Instruct,BD1,one,14.002,True,8
2,tiiuae/Falcon3-7B-Instruct,BD1,few,14.805,True,9
3,tiiuae/Falcon3-7B-Instruct,BD1,many,14.858,True,9
4,tiiuae/Falcon3-7B-Instruct,BD2,zero,20.094,False,7
5,tiiuae/Falcon3-7B-Instruct,BD2,one,18.573,True,9
6,tiiuae/Falcon3-7B-Instruct,BD2,few,19.488,True,9
7,tiiuae/Falcon3-7B-Instruct,BD2,many,20.021,True,9
8,Qwen/Qwen2.5-14B-Instruct,BD1,zero,15.637,True,9
9,Qwen/Qwen2.5-14B-Instruct,BD1,one,16.719,True,9


In [3]:
for row in rows:
    print('\n'+'='*80)
    print(row['model_id'],row['input_id'],row['shot_mode'])
    print('Seconds:',row['generation_seconds'],'Tokens:',row['output_tokens'],'Schema:',row['schema_valid'])
    print('Validation:',row['validation_errors'] or 'Passed shared schema')
    print(row['raw_output'])
    print('Quality:',row['quality_score_0_to_10'],'/10')
    print(row['review_notes'])



tiiuae/Falcon3-7B-Instruct BD1 zero
Seconds: 17.783 Tokens: 184 Schema: True
Validation: Passed shared schema
```json
{
  "tasks": [
    {
      "title": "Clean the kitchen",
      "task_type": "CHORE",
      "priority": "MEDIUM",
      "description": "Prepare the kitchen for guests arriving tomorrow."
    },
    {
      "title": "Go shopping",
      "task_type": "SHOPPING",
      "priority": "URGENT",
      "description": "Buy groceries needed for tomorrow's guests."
    },
    {
      "title": "Fix the bathroom tap",
      "task_type": "MAINTENANCE",
      "priority": "URGENT",
      "description": "The tap is leaking badly and needs immediate attention."
    }
  ]
}
```
Quality: 8 /10
Assistant-assisted review; E/T/P/C/F=3/2/0/2/1. Three distinct concerns with correct types. Kitchen MEDIUM understates tomorrow's preparation and tap URGENT assumes an immediate safety risk; grocery urgency may reflect the next-day deadline. Fenced JSON passes the shared contract.

tiiuae/Falcon3-7B-I

In [4]:
display(Markdown((ROOT/'docs/a6-person-4/large-models.md').read_text(encoding='utf-8')))
display(Markdown((ROOT/'docs/a6-person-4/local-vs-api.md').read_text(encoding='utf-8')))


# P4 larger-category correction

These tests complete P4's requested extension to the original large/above-10B
roster. During the run, main advanced to `5afd378` (PR #31): P1 adopted different
balanced size bands and kept the earlier P4 small-model tests in its selected
15-model notebook. That merged selection is preserved. The new results are
separate evidence; they are not silently added to P1's 15 selected models.

The correction tests these four public models:

| Model | Intended category | Cases |
| --- | --- | --- |
| `tiiuae/Falcon3-7B-Instruct` | Large | BD1/BD2 × zero/one/few/many |
| `Qwen/Qwen2.5-14B-Instruct` | Above 10B | BD1/BD2 × zero/one/few/many |
| `microsoft/Phi-3-medium-4k-instruct` | Above 10B | BD1/BD2 × zero/one/few/many |
| `Qwen/Qwen3-14B` | Above 10B | BD1/BD2 × zero/one/few/many |

Qwen3-14B was selected by the user as the additional above-10B model. P1 owns
the final category plan and final notebook. These results supply one-large and
three-above-10B candidates under the original ranges (≤0.5B, >0.5–2B, >2–4B,
>4–10B, >10B). PR #31 instead uses <0.2B, 0.2–<0.6B, 0.6–<1.6B, 1.6–<3.5B,
and ≥3.5B. Deciding which category plan to submit remains P1/the team's work;
P4 does not claim these different plans are interchangeable.

## Reproduction and evidence

Open [person4_large_models_colab.ipynb](../../llm-test/person4_large_models_colab.ipynb)
in Colab, select an available GPU, and run all cells. No paid compute purchase,
API key, Hugging Face token, or Drive mount is needed. The notebook embeds
[person4_large_benchmark.py](../../llm-test/person4_large_benchmark.py).
Its separate dependency list is
[requirements-person4-large.txt](../../llm-test/requirements-person4-large.txt);
the Django dependency file is unchanged.

The runner verifies a SHA-256-pinned copy of the team's existing prompt and
validation helpers from main commit `14a02aeb4eb71d1ddf7dfb66d1e868b77213eb28`.
It runs actual Transformers inference, pins each model revision, uses greedy
generation with 220 new tokens, and disables Qwen3 thinking. Weights use NF4
double quantization with FP32 computation and FP32 non-quantized modules;
attention is eager. These are quantized measurements, not native FP16 results.

Each response checkpoints an eleven-column CSV under
`a6_results/person4_large/person4_large_results.csv`. Metadata records the GPU,
versions, revisions, configuration, load time, and the last completed case's
peak allocated GPU memory (not the maximum across all cases).
Load time includes first downloads when uncached. Generation timing synchronizes
CUDA. Cross-hardware and cross-precision speed comparisons are not controlled.

The notebook uses a separate process per model and removes only that model's
dedicated downloaded weight cache after success. It exports a ZIP of the
runner, contract, CSV, and metadata, excluding weights. If a model fails, export
the partial evidence and inspect the error; partial rows are not a completed
benchmark. Re-running resumes completed input/shot cases.

Quality scores remain blank until the raw outputs have been read. Apply the
same extraction/types/priority/clarity/format rubric (3/2/2/2/1), with explicit
per-response reasons and review attribution. Schema validity alone does not
establish task quality. Do not invent scores or imply independent human review.

## Completed execution and handoff

All four models completed on October 18, 2026 (CDT; October 19 UTC), on a Tesla
T4 reporting 14.56 GiB VRAM. All **32 unique generations** were exported and
validated locally against the exact shared parser/schema. The exported runner
matched the repository source. ZIP SHA-256:
`f471c3e53964b86bf6b320b0744a4eb83ac75ac9f97a45287954a30dd6419cf5`.
These are cloud GPU model runs in the user's Colab, not measurements on the
Windows laptop or an on-premise privacy test. Inputs were the two fixed examples.

- New P4 input for any larger-model consolidation: [person4_large_results.csv](../../a6_results/person4_large/person4_large_results.csv).
- Full raw outputs and explicit review notes: [results notebook](../../llm-test/person4_large_results_review.ipynb).
- Findings: [large-model-findings.md](large-model-findings.md).
- Generation CSV/metadata are preserved separately as `person4_large_raw_results.csv`
  and `person4_large_run_metadata.json`; the review audit records their hashes.
  A directory-local `.gitattributes` keeps these files and the pinned contract
  in LF form so Windows checkout conversion cannot invalidate the hashes.
- [Colab completion screenshot](large-model-colab-complete.png) records the
  final 32-generation/eight-per-model count.

**If the team returns to the original ranges**, P1 should use the new 32-row CSV
instead of `a6_results/person4_benchmark_results.csv`, retain P2/P3, and select
three of P1's four ≤0.5B models. That yields 15 models/120 generations and
3/3/3/3/3 under the original ranges. Keeping PR #31's newer bands instead means
these four runs remain supplemental. Do not blindly concatenate both P4 sets.
P1's final notebook was not edited by P4.

The 32 quality reviews are assistant-assisted judgments with explicit criterion
points, not independent human signoff. Priorities remain context-sensitive;
read the audit's calibration notes. Schema validity is not semantic correctness.

```powershell
python docs/a6-person-4/verify_large_results.py
# Reapply only the frozen reviews for these unchanged raw outputs:
python docs/a6-person-4/review_large_results.py
python docs/a6-person-4/build_large_report.py
python docs/a6-person-4/execute_notebook.py --notebook llm-test/person4_large_results_review.ipynb
```

The benchmark harness passed five unit tests. The full Django suite passed 175
tests and its system check reported no issues. The results notebook is executed
against the exported CSV; it does not pretend to rerun GPU inference locally.


# Part 3 — P4: Local model vs paid API

Pricing checked October 18, 2026. These are transparent planning estimates, not
paid API experiments or proof that different-sized models have equal quality.
P4's historical CPU findings are in `benchmark_summary.md`; the completed 7–14B
GPU results are in [large-model-findings.md](large-model-findings.md) and the
[32-run CSV](../../a6_results/person4_large/person4_large_results.csv).

## Workload and cost assumptions

Assume 10,000 daily users, two extractions per user per day, 800 input tokens per
extraction (including instructions/examples) and 220 output tokens. That is
20,000 requests, 16M input tokens, and 4.4M output tokens per day. Actual prompt
length varies by shot mode; 800 is a planning scenario, not a measured average.
Thirty days gives 600,000 requests, 480M input and 132M output tokens.

Groq's published GPT-OSS 20B prices are $0.075/M input tokens and $0.30/M output
tokens. GPT-OSS 120B is $0.15/M input and $0.60/M output. These are public text
inference price references, not the team's selected Person 3 provider or a claim
of equivalence to the three sub-billion local models.
[Source: Groq supported models](https://console.groq.com/docs/models).

| Option | Calculation | Estimated USD |
| --- | --- | ---: |
| API 20B, per extraction | (800 × 0.075 + 220 × 0.30) / 1,000,000 | $0.000126 |
| API 20B, daily | 16 × 0.075 + 4.4 × 0.30 | $2.52 |
| API 20B, 30 days | 480 × 0.075 + 132 × 0.30 | $75.60 |
| API 120B, 30 days | 480 × 0.15 + 132 × 0.60 | $151.20 |
| Example RTX 3090 always on | $0.50/hour × 24 × 30 | $360.00 |
| Example RTX 4090 always on | $0.89/hour × 24 × 30 | $640.80 |

Runpod's displayed Pod configurations list RTX 3090 (24 GB VRAM, 125 GB host RAM,
16 vCPUs) at $0.50/hour and RTX 4090 (24 GB VRAM, 41 GB host RAM, 6 vCPUs) at
$0.89/hour. They are example offers, not quotes for the assignment's exact
16-core/64-GB/1-TB configuration. Availability, storage, network, replicas,
monitoring, application hosting, and engineering are additional considerations.
[Source: Runpod pricing](https://www.runpod.io/pricing).

Free tiers and student credits can reduce classroom costs, but rate limits and
expiry make them unsuitable as the only 10,000-user capacity plan. Cached input
discounts, batching, tax, retries and reasoning tokens are omitted. Include all
billed tokens in a real budget. We made no paid API calls for this comparison.

## When a local server is acceptable

Local inference can fit a classroom demonstration, an already-owned machine,
steady predictable demand, offline operation, or a documented requirement to
keep data on infrastructure controlled by the team. Existing hardware has a
marginal electricity cost rather than a new cloud rental bill, but still needs
maintenance and capacity planning. Local inference itself has no per-token bill.

For the assignment's hypothetical 16-core CPU, GPU, 64 GB RAM, and 1 TB disk,
specify the GPU model and **VRAM**: “16-core GPU” alone does not describe model
capacity. Approximate unquantized weights consume 4 bytes/parameter in FP32 and
2 bytes in FP16/BF16; a 7B model is about 28/14 GB, a 14B model about 56/28 GB.
Runtime activations, KV cache, Python, concurrent requests and the OS need extra
memory. A 14B FP16 model therefore exceeds a single 24-GB GPU before overhead.
Four-bit weights are about 0.5 byte/parameter before quantization metadata, but
require supported kernels and a separate quality/latency evaluation. Host RAM is
not interchangeable with GPU VRAM; disk capacity does not make inference fast.

The earlier small-model runs used CPU FP32 on approximately 15.5 GiB RAM.
P4 has now also measured Falcon3-7B and three 14B candidates on a Colab Tesla T4
with NF4 weights and FP32 computation/non-quantized modules. Qwen2.5-14B passed
the shared schema in 8/8 cases, averaged 20.778 seconds of generation, and scored
9.5/10 in assistant-assisted content review. Qwen3-14B also passed 8/8, averaged
21.801 seconds, and scored 9.25/10. Falcon passed 7/8; Phi passed 3/8 and had
schema mismatches/truncation under the shared 220-token cap. Human review remains
necessary even for valid JSON. These are two fixed inputs per model, not a
production quality or load test; speed comparisons with CPU/native runs are
not controlled. Main PR #31 uses a different balanced 15-model selection;
P1/the team decides whether to extend that analysis with these above-10B runs.

The Qwen2.5-14B last-case peak allocated VRAM was 12.751 GiB on the 14.563-GiB
T4; this field is not an aggregate maximum over every case. Quantization made
this test feasible, but does not establish the capacity of the hypothetical
server. Colab is cloud-hosted model inference, not proof of on-premise/offline
privacy. Only the fixed synthetic test paragraphs were used.

## When an API is preferable

At this illustrative workload, $75.60/month in tokens is below a continuously
rented $360/month GPU, before either option's common Django costs. APIs are
attractive for spiky demand, quick rollout, stronger-model quality, and avoiding
model loading/serving operations, provided acceptable privacy terms, rate limits,
latency and provider reliability are established. A published model size alone
does not prove comparable extraction quality; rerun the same prompts on any
candidate paid model before selecting it.

Using the 20B price example, a $12/day local rental equals token spending at about
95,238 extractions/day: 12 / 0.000126. This is a **billing-only** break-even. It
does not show that one GPU can serve that workload. At 20,000/day, average arrival
rate is 0.2315 requests/second. If local measured mean generation time is L
seconds, serialized service needs at least 0.2315 × L continuously busy workers
before prompt processing, peak headroom and latency constraints. At five times
average peak demand, multiply that by five. Use a production inference server,
queue, bounded concurrency and a real load test rather than running many Django
workers that each load an independent model.

For scale only, applying the observed Qwen2.5-14B generation mean to 20,000
requests gives 20,000 × 20.778 / 3,600 = **115.43 serialized GPU-hours/day**,
or **4.81 fully occupied T4 equivalents** before peak headroom. This is arithmetic
on a two-input sample, not a demonstrated serving capacity: the measured token
counts differ from the planning 800/220 scenario. Batching, prompt processing,
concurrency, model loading and end-to-end latency need an actual serving test.
Five Django workers do not turn one T4 into five independent GPU replicas.

## Hybrid potential

Use the local model to extract ordinary household tasks. Validate output and
require user review; reserve external AI for an explicitly requested breakdown
or a user-approved retry when local extraction fails. Never send raw household
input to an external provider silently. A quality gate can detect malformed
schema but cannot reliably detect all semantic mistakes; do not treat it as a
calibrated model-confidence score.

Example: 20% of the 20,000 daily requests use API extraction, and 10% request a
breakdown costing 300 input/600 output tokens. API extraction costs $0.504/day;
2,000 breakdowns cost (0.6 × 0.075 + 1.2 × 0.30) = $0.405/day. Combined external
cost is $0.909/day, or $27.27/30 days, **plus local hosting**. Adding a rented
$360/month GPU totals $387.27, which exceeds the $75.60 all-API extraction-only
scenario; those workloads differ because the hybrid also includes breakdowns.
Including the same breakdown volume in all-API yields $87.75/month. Hybrid is
therefore not automatically cheaper; it can make sense with already-owned local
capacity, privacy constraints, sustained utilization, or expensive API models.

The merged implementation reaches Person 3's Groq action-plan feature from a
saved task's detail page, after Brain Dump review and confirmation. A separate
disclosed POST sends the saved task fields and returns an unsaved plan preview.
P4's earlier pre-save callable remains an optional, disabled extension and is
not needed for this supported flow. Private Groq configuration is required for
live use; Person 3's live-call evidence is stored separately. P4's combined-path
test mocks the external HTTP response. The prices above remain planning estimates,
not P4 measurements of paid-provider quality or latency.
